# Phase 3 Worksheet — Embeddings: In-house vs Hugging Face Models
`pip install sentence-transformers --break-system-packages` — this will download models from Hugging Face on first use, per your go-ahead for this phase.

**Corrected in this version:** in-house embedding calls go through `embedder.embed_query()` instead of `get_embedding(text, model=MODEL_JINA)`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))  # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=200):
    """Drop-in replacement for the old multimodal_chat() text-only calls --
    correctly routed per-model via get_chat_model(), unlike inhouse_llm.py's
    own chat()/multimodal_chat() which always hit the Qwen3-14B endpoint."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=200):
    """Drop-in replacement for multimodal_chat() WITH an image -- uses the
    corrected image_url content-block format, and an actual client for the
    vision model (inhouse_llm.py never created one)."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

import chromadb
client = chromadb.HttpClient(host="localhost", port=8000)  # adjust to your Chroma server
print("Setup OK")

## 1. Similarity metrics by hand

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

v1 = embedder.embed_query("What is RAG?")
v2 = embedder.embed_query("Explain retrieval-augmented generation")
print("Jina cosine similarity:", cosine(v1, v2))

## 2. Loading Hugging Face embedding models directly
(unchanged — these are separate from the in-house wrapper by design)

In [ ]:
from sentence_transformers import SentenceTransformer

bge = SentenceTransformer("BAAI/bge-large-en-v1.5")
nomic = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
minilm = SentenceTransformer("all-MiniLM-L6-v2")

text = "What is RAG?"
print("BGE dim:", len(bge.encode(text)))
print("Nomic dim:", len(nomic.encode(text)))
print("MiniLM dim:", len(minilm.encode(text)))
print("Jina dim:", len(embedder.embed_query(text)))

## 3. Reproducing this phase's teaser bug: mixing models in one collection

In [ ]:
bad_collection = client.get_or_create_collection("phase3_mixed_bug")

texts = ["MCP standardizes tool calling.", "RAG grounds answers in context."]
jina_vec = embedder.embed_query(texts[0])
bge_vec = bge.encode(texts[1]).tolist()

# deliberately mixing incompatible vector spaces in one collection
bad_collection.upsert(ids=["jina_doc", "bge_doc"], embeddings=[jina_vec, bge_vec], documents=texts)

query_vec = embedder.embed_query("How do LLMs call tools?")  # embedded with JINA
result = bad_collection.query(query_embeddings=[query_vec], n_results=2)
print("Mixed-collection query result (distances are not meaningfully comparable):")
print(result["documents"][0], result["distances"][0])

## 4. The fix: separate collections per model

In [ ]:
coll_jina = client.get_or_create_collection("phase3_docs_jina")
coll_bge = client.get_or_create_collection("phase3_docs_bge")

coll_jina.upsert(ids=["d1"], embeddings=[embedder.embed_query(texts[0])], documents=[texts[0]])
coll_bge.upsert(ids=["d2"], embeddings=[bge.encode(texts[1]).tolist()], documents=[texts[1]])

q_jina = embedder.embed_query("How do LLMs call tools?")
q_bge = bge.encode("How do LLMs call tools?").tolist()

print("Jina collection result:", coll_jina.query(query_embeddings=[q_jina], n_results=1)["documents"])
print("BGE collection result:", coll_bge.query(query_embeddings=[q_bge], n_results=1)["documents"])

## Teaser exercise
Embed the same 10 sentences with Jina, BGE, and MiniLM into 3 separate collections. Run the same 5 queries against all 3. Do all three models agree on the top hit every time, or do you find cases where they disagree? Disagreements are exactly where model choice would matter on a real project.